In [1]:
# lectura de los datos

import pandas as pd
import read_data

# Suponiendo que el activo tiene ese nombre
# solo es remplazar por el nombre que se tenga
# solo se acepta archivos .parquet y .csv
nombre_activo: str = "EURUSD_year.parquet"
data: pd.DataFrame = read_data.read_asset(nombre_activo)

In [2]:
data

,Precio Spot
time,
2026-02-02 00:01:00.171,1.18516
2026-02-02 00:01:00.237,1.18531
2026-02-02 00:01:22.175,1.18529
2026-02-02 00:01:30.177,1.18519
2026-02-02 00:01:36.259,1.18520
...,...
2026-03-06 23:54:59.204,1.16132
2026-03-06 23:54:59.531,1.16131
2026-03-06 23:54:59.681,1.16135


In [3]:
first_date = data.index[0].strftime('%Y-%m')
last_date = data.index[-1].strftime('%Y-%m')

print(f"inicia en {first_date} y termina en {last_date}")

inicia en 2026-02 y termina en 2026-03


In [ ]:
import find_best
import keys
import pandas as pd
from use_tecnics import simple_methods

keys.candles = 10
keys.methods = simple_methods

ventana_entrenamiento = pd.Timedelta(weeks=4)
ventana_testeo = pd.Timedelta(weeks=1)

inicio_train = data.index[0].normalize()
fin_datos = data.index[-1]

while True:
    fin_train = inicio_train + ventana_entrenamiento
    fin_test = fin_train + ventana_testeo

    if fin_train >= fin_datos:
        break

    train_data = data.loc[inicio_train : fin_train - pd.Timedelta(minutes=1)]
    test_data = data.loc[fin_train : fin_test - pd.Timedelta(minutes=1)]

    if test_data.empty:
        break

    print(f"Entrenando MA (4 semanas): {inicio_train.strftime('%Y-%m-%d')} a {fin_train.strftime('%Y-%m-%d')}")

    mans: list = find_best.opti_main(train_data)

    vela = mans[1]
    lookback = mans[2] 

    minutos_necesarios = vela * lookback
    colchon = train_data.iloc[-minutos_necesarios:]
    test_data_con_colchon = pd.concat([colchon, test_data])

    print(f"Resultados del testeo FUERA DE MUESTRA (1 semana)")
    print(f"Operando desde: {test_data.index[0].strftime('%Y-%m-%d %H:%M')}")
    print(f"Hasta: {test_data.index[-1].strftime('%Y-%m-%d %H:%M')}")

    find_best.read_results(mans, test_data_con_colchon)

    inicio_train = inicio_train + ventana_testeo

Entrenando MA (4 semanas): 2025-02-21 a 2025-03-21
